### 건물통합정보 + kepco_final_data.csv
건물 통합 정보에 용도 매핑해서 건물별로 연평균 전력 사용량 매칭<br>
히트맵으로 용도별로 전력 사용량 색이 다르게 나타나는지 확인

**GIS건물통합정보**
https://www.vworld.kr/dtmk/dtmk_ntads_s002.do?svcCde=NA&dsId=18

#### 건물 통합 정보 전처리

In [1]:
import geopandas as gpd
import pandas as pd

gdf_raw = gpd.read_file('../data/AL_D010_11_20260609.shp', encoding='cp949')

# 서울 필터
gdf_seoul = gdf_raw[gdf_raw['A4'].str.startswith('서울')].copy()

# 좌표계 변환
gdf_seoul = gdf_seoul.to_crs('EPSG:4326')

# 결측 처리
gdf_seoul = gdf_seoul.dropna(subset=['A9']).copy() # 건물용도


gdf_seoul["시군구"] = gdf_seoul["A4"].str.extract(r"([^ ]+구)")

# 건물용도 매핑 (kepco 계약종별 -> 전력용도)
mapping = {

    # ======================
    # 🏠 주택용
    # ======================
    "단독주택": "주택용",
    "공동주택": "주택용",
    "다가구주택": "주택용",

    # ======================
    # 🏢 일반용 (상업/서비스/공공)
    # ======================
    "제1종근린생활시설": "일반용",
    "제2종근린생활시설": "일반용",
    "업무시설": "일반용",
    "의료시설": "일반용",
    "노유자시설": "일반용",
    "종교시설": "일반용",
    "문화및집회시설": "일반용",
    "판매시설": "일반용",
    "판매및영업시설": "일반용",
    "위락시설": "일반용",
    "관광휴게시설": "일반용",
    "숙박시설": "일반용",
    "운동시설": "일반용",
    "근린생활시설": "일반용",
    "공공용시설": "일반용",
    "장례식장": "일반용",
    "교육연구및복지시설": "일반용",
    "방송통신시설": "일반용",
    "운수시설": "일반용",
    "수련시설": "일반용",
    "교정및군사시설": "일반용",

    # ======================
    # 🎓 교육용
    # ======================
    "교육연구시설": "교육용",

    # ======================
    # 🏭 산업용
    # ======================
    "공장": "산업용",
    "창고시설": "산업용",
    "자동차관련시설": "산업용",
    "위험물저장및처리시설": "산업용",
    "동.식물 관련시설": "산업용",
    "발전시설": "산업용",
    "분뇨.쓰레기처리시설": "산업용",
    "묘지관련시설": "산업용",
    "가설건축물": "산업용",

    # ======================
    # 🌾 농사용 (거의 제한적)
    # ======================
    # 실제 데이터에 거의 없음, 동식물 관련시설 일부 포함 가능
}

gdf_seoul["전력용도"] = gdf_seoul["A9"].map(mapping)

print(f'완료: {len(gdf_seoul)}개 건물')
print(f'전력용도 null 개수: {gdf_seoul["전력용도"].isnull().sum()}') #null이면 mapping 안된 것

완료: 546171개 건물
전력용도 null 개수: 0


#### 빌딩 필요 데이터만 필터

In [5]:
building = gdf_seoul[["A4","시군구", "A9", "전력용도", "geometry"]].copy()
building.columns = ["adress", "sigungu", "type", "elec_type", "geometry"]
building.head()

,adress,sigungu,type,elec_type,geometry
0,서울특별시 종로구 숭인동,종로구,단독주택,주택용,"POLYGON ((127.02166 37.57732, 127.02165 37.577..."
1,서울특별시 종로구 숭인동,종로구,단독주택,주택용,"POLYGON ((127.016 37.57725, 127.01596 37.57725..."
2,서울특별시 종로구 숭인동,종로구,단독주택,주택용,"POLYGON ((127.01633 37.57723, 127.01633 37.577..."
3,서울특별시 종로구 숭인동,종로구,단독주택,주택용,"POLYGON ((127.01651 37.57717, 127.01646 37.577..."
5,서울특별시 종로구 이화동,종로구,제2종근린생활시설,일반용,"POLYGON ((127.00532 37.57728, 127.00534 37.577..."


#### 건물 통합 정보 + kepco merge -> json

In [7]:
risk_df = pd.read_csv('../data/output/kepco_final_data.csv')
risk_df.head()

,연도,시도,시군구,전력용도,월,전력사용량,연도_월별_시군구별_전력사용량합계,전력사용율,공급예비율(%),공급위험도,용도정규화사용률,수요감축필요도_1st
0,2006,서울특별시,강남구,가로등,1,3089,390018,0.792,25.690,0.74310,0.010712,0.007960
1,2006,서울특별시,강남구,가로등,2,2843,357912,0.794,21.943,0.78057,0.010763,0.008401
2,2006,서울특별시,강남구,가로등,3,2970,338933,0.876,17.674,0.82326,0.012035,0.009908
3,2006,서울특별시,강남구,가로등,4,2608,331685,0.786,21.966,0.78034,0.010879,0.008489
4,2006,서울특별시,강남구,가로등,5,2281,321151,0.710,29.926,0.70074,0.009773,0.006848


In [ ]:
# elecdemand_preprocessing.ipynb 에서 생성한 kepco_final_data.csv 읽기
risk_df = pd.read_csv('../data/output/kepco_final_data.csv')

# =========================
# 1. 문자열 정리
# =========================
gdf_seoul["전력용도"] = gdf_seoul["전력용도"].str.strip()

risk_df["시군구"] = risk_df["시군구"].str.strip()
risk_df["전력용도"] = risk_df["전력용도"].str.strip()

gdf_seoul = gdf_seoul.drop(columns=["A22"])

# =========================
# 2. Timestamp / dtype 정리 (핵심 해결)
# =========================
risk_df["연도"] = risk_df["연도"].astype(int)
risk_df["월"] = risk_df["월"].astype(int)

# =========================
# 3. 시계열 packing
# =========================
time_series = risk_df.groupby(
    ["시군구", "전력용도"]
).apply(lambda df: df[[
    "연도",
    "월",
    "전력사용량",
    "연도_월별_전력사용량합계",
    "전력사용율",
    "공급예비율(%)",
    "공급위험도",
    "용도정규화사용률",
    "용도가중치",
    "위험도지수",
    "등급"
]].to_dict("records")
).reset_index()

time_series.columns = ["시군구", "전력용도", "time_series"]

# =========================
# 4. merge
# =========================
merged = gdf_seoul.merge(
    time_series,
    on=["시군구", "전력용도"],
    how="left"
)

# =========================
# 5. GeoDataFrame
# =========================
gdf = gpd.GeoDataFrame(
    merged,
    geometry="geometry",
    crs="EPSG:4326"
)

# =========================
# 6. NaN 안전 처리
# =========================
gdf["time_series"] = gdf["time_series"].apply(
    lambda x: x if isinstance(x, list) else []
)

# =========================
# 7. GeoJSON 생성 (FIXED)
# =========================
geojson = gdf.to_json()
# gdf.to_file('output/building_power.geojson', driver = 'GeoJSON')


# (선택) 파일 저장
with open("output/building_elec.geojson", "w", encoding="utf-8") as f:
    f.write(geojson)